# AGH-Former v2 - Google Colab Pro
Sade 2 aşamalı yapı: arşivlenmiş vNext global aşaması + tek All23-MVSC yerel düzeltici.
Sırayla çalıştırın: kurulum → smoke → 5 fold validasyon → (kapı geçerse) final test.

Gerekli: `agh_vnext_runs/publication_cv_seed42` (vNext arşivi) ve `all23_rgb_geodesic_runs/publication_cv_stage1_v4_seed42` (ön işleme) Drive'da olmalı.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess
CODE_ROOT = Path('/content/comparative-study')
REPO_URL = 'https://github.com/eckdev/comparative-study.git'
if CODE_ROOT.exists():
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(CODE_ROOT)], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(CODE_ROOT / 'agh_former_vnext_orthodontic_comparison/requirements.txt')], check=True)

In [ ]:
STAGE_A = Path('/content/drive/MyDrive/orthodontic/agh_vnext_runs/publication_cv_seed42')
PRE = Path('/content/drive/MyDrive/orthodontic/all23_rgb_geodesic_runs/publication_cv_stage1_v4_seed42')
for fold in range(1, 6):
    assert (STAGE_A / f'fold_{fold}/predictions_val.csv').exists(), fold
    assert (PRE / f'fold_{fold}').exists(), fold
print('Arşiv ve ön işleme bulundu')

## 1. Smoke (Fold 1, 2 epoch, yalnızca validasyon)

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2_smoke --seed 42

## 2. Beş fold validasyon
Test dosyası açılmaz. Sonunda `validation_gate.json` yazılır.

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2_val_all_folds --seed 42

In [ ]:
import json
gate = json.loads(Path('/content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_seed42/validation_gate.json').read_text())
print(json.dumps({k: gate[k] for k in ('v2_validation', 'vnext_validation', 'checks', 'run_final_test')}, indent=2))

## 2b. Kapı yeniden hesaplama + tanı raporu (GPU gerekmez)
Kapıyı (fold, kişi) bazında yeniden hesaplar ve `validation_diagnostics/diagnostics_val.md` yazar. Test dosyası okunmaz.

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2_diagnostics --seed 42

In [ ]:
from IPython.display import Markdown
Markdown(Path('/content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_seed42/validation_diagnostics/diagnostics_val.md').read_text())

## 3. Final test (yalnızca kapı geçtiyse, bir kez)

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2_final_test --seed 42

In [ ]:
%cd /content/comparative-study
!python -m shared_metrics.aggregate_cv_results --run-dir /content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_seed42 --model 'AGH-Former v2'

# v2.1
Aşama A = vNext Stage 2 gate'li çıktı; v2'nin eğittiği düzelticiler yeniden kullanılır (eğitim yok), ensemble + yeni gate sinyalleri + tip bazında iterasyon.
Önkoşul: `agh_v2_runs/publication_cv_seed42` (v2 koşusu) Drive'da tam olmalı.

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v21_val_all_folds --seed 42

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v21_diagnostics --seed 42

In [ ]:
from IPython.display import Markdown
Markdown(Path('/content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_v21_seed42/validation_diagnostics/diagnostics_val.md').read_text())

## v2.1 final test (yalnızca kapı geçtiyse, bir kez)

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v21_final_test --seed 42

In [ ]:
%cd /content/comparative-study
!python -m shared_metrics.aggregate_cv_results --run-dir /content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_v21_seed42 --model 'AGH-Former v2.1'

# Final model: gate'siz v2
Ayrıntılar `agh_former_v2/PREREGISTRATION_TR.md` dosyasında. Sırayla çalıştırın. **4. adım (final test) bir kez çalışır.**

## 1. Validasyon (yalnızca çıkarım)

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2final_val_all_folds --seed 42

## 2. Tanı raporu ve yeniden üretim kontrolü
Beklenen değerler: all-23 ≈ 1.994, Core20 ≈ 1.578, Hard3 ≈ 4.762. Belirgin fark varsa 3. adıma geçmeyin.

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2final_diagnostics --seed 42

## 3. Final model kararını kaydet (testten önce)

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2final_commit --seed 42

## 4. Final test (bir kez)

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2final_final_test --seed 42

In [ ]:
%cd /content/comparative-study
!python -m shared_metrics.aggregate_cv_results --run-dir /content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_v2final_seed42 --model 'AGH-Former v2'

# Değişiklik: yalnızca mesh, katılımcı bilgisi yok (v2final_meshonly)
Uzamsal prior hiçbir katılımcı bilgisi (sınıf, cinsiyet) kullanmaz; eğitilmiş modeller aynen kullanılır (eğitim yok). Ayrıntı: `PREREGISTRATION_TR.md` → Değişiklik kaydı.

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2final_meshonly_val_all_folds --seed 42

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2final_meshonly_diagnostics --seed 42

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2final_meshonly_commit --seed 42

In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/colab_run_agh_v2.py --preset v2final_meshonly_final_test --seed 42

In [ ]:
%cd /content/comparative-study
!python -m shared_metrics.aggregate_cv_results --run-dir /content/drive/MyDrive/orthodontic/agh_v2_runs/publication_cv_v2final_meshonly_seed42 --model 'AGH-Former v2'

# Hesaplama maliyeti (makale, Tablo 5)
Dört model fold 1'in 60 test taramasında **tek tek** (batch 1) çalıştırılır; önbellekler kapalıdır. Ölçülenler: parametre sayısı, tarama başına süre (CPU önişleme + ağ) ve tepe GPU belleği. Üretilen koordinatlar arşivdeki tahminlerle karşılaştırılır (`max_difference_to_archived_mm` ≈ 0 olmalı). Eğitim yapılmaz, test sonuçları değişmez. A100 GPU seçin; süre yaklaşık 30–60 dakika.

In [ ]:
subprocess.run(['pip', 'install', '-q', 'robust-laplacian', 'potpourri3d'], check=True)
%cd /content/comparative-study
!python -u agh_former_v2/measure_cost.py --fold 1 --out /content/drive/MyDrive/orthodontic/cost/cost_fold1.json

In [ ]:
import json
cost = json.loads(Path('/content/drive/MyDrive/orthodontic/cost/cost_fold1.json').read_text())
print(json.dumps({'environment': cost['environment'], 'models': {k: {key: v[key] for key in v if key != 'per_scan'} for k, v in cost['models'].items()}}, indent=2))

# Ablasyon: düzelticinin bileşenleri
Stage 1, Stage 2 ve shape prior arşivden okunur; yalnızca düzeltici değişir ve her varyant son modelin kilitli politikasıyla uygulanır.
- **I0–I4** (eğitim yok): tam topluluk, tek düzeltici, renk kapalı, Stage 1'den ve Stage 2'den başlama.
- **R0–R5** (fold başına bir düzeltici eğitilir): R0 değişikliksiz yeniden eğitim (referans), R1 görüntüsüz, R2 bağlam görünümsüz, R3 ortalama füzyon, R4 yalnızca görüntü, R5 renksiz.

Çıktılar `orthodontic/ablation/<varyant>/fold_k/` altına yazılır; biten varyantlar atlanır, oturum koparsa hücreyi yeniden çalıştırın. A100 GPU seçin.

**Önce 1. adım:** fold 1'de I0 arşivi yeniden üretmeli (ALE ≈ 2.0, `max difference` birkaç mm'yi geçmemeli) ve R0'ın eğitim süresi görülmeli. Tüm çalıştırmaya bu süreye bakarak karar verin.


In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/run_ablation.py --folds 1 --variants I0_full,I1_single,R0_retrained


In [ ]:
%cd /content/comparative-study
!python -u agh_former_v2/run_ablation.py --folds 1,2,3,4,5 --variants all


In [ ]:
import json
ABL = Path('/content/drive/MyDrive/orthodontic/ablation')
for report in sorted(ABL.glob('*/fold_*/report.json')):
    r = json.loads(report.read_text())
    print(f"{r['variant']:<17} fold {r['fold']}  ALE {r['ale']:.3f}  max diff {r['max_difference_to_archived_final_mm']:.2f} mm  train {r['training_seconds']}")
